In [2]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [3]:
!pip install wandb -q

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key

# This logs you in automatically without prompting for a password
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kapilsandhu2022 (kapilsandhu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
%%writefile eval_nshot_baseline.py
import os
import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator
from sklearn.metrics import classification_report, accuracy_score
import wandb 

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ============================================================
# SECTION 1: Configuration
# ============================================================
MODEL_NAME    = "Qwen/Qwen2.5-0.5B"
DATASET_NAME  = "stanfordnlp/snli"
WANDB_PROJECT = "qwen-snli-finetuning" 
BATCH_SIZE    = 32  # Per GPU (Effective = 64)

# Label Mapping
LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}
ID2LABEL = {0: "entailment", 1: "neutral", 2: "contradiction"}

# ============================================================
# SECTION 2: Authentic SNLI Prompts
# ============================================================
SYSTEM_PROMPT = "Determine the logical relationship between the premise and the hypothesis. Respond with exactly one word: 'entailment', 'neutral', or 'contradiction'.\n\n"

SHOT_1 = (
    "Premise: A person on a horse jumps over a broken down airplane.\n"
    "Hypothesis: A person is outdoors, on a horse.\n"
    "Relationship: entailment\n\n"
)

SHOT_3 = SHOT_1 + (
    "Premise: A person on a horse jumps over a broken down airplane.\n"
    "Hypothesis: A person is training his horse for a competition.\n"
    "Relationship: neutral\n\n"
    "Premise: A person on a horse jumps over a broken down airplane.\n"
    "Hypothesis: A person is at a diner, ordering an omelette.\n"
    "Relationship: contradiction\n\n"
)

def build_prompt(premise, hypothesis, mode="0-shot"):
    base = f"Premise: {premise}\nHypothesis: {hypothesis}\nRelationship:"
    if mode == "0-shot": return SYSTEM_PROMPT + base
    if mode == "1-shot": return SYSTEM_PROMPT + SHOT_1 + base
    if mode == "3-shot": return SYSTEM_PROMPT + SHOT_3 + base

# ============================================================
# SECTION 3: Main DDP Evaluation Loop
# ============================================================
def main():
    accelerator = Accelerator(log_with="wandb")
    accelerator.print(f"GPUs active: {accelerator.num_processes}")
    
    accelerator.init_trackers(
        project_name=WANDB_PROJECT,
        config={
            "model_name": MODEL_NAME,
            "dataset": DATASET_NAME,
            "batch_size_per_gpu": BATCH_SIZE,
            "run_type": "N-Shot Baseline Evaluation"
        }
    )
    
    # 1. Load Data (FULL TEST SET)
    with accelerator.main_process_first():
        accelerator.print("\nLoading FULL SNLI Test Set (~10,000 samples)...")
        dataset = load_dataset(DATASET_NAME, split="test")
        dataset = dataset.filter(lambda x: x["label"] != -1)

    # 2. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.padding_side = "left" 
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 3. Load Generative Model
    accelerator.print("\nLoading Base Generative Model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, 
        trust_remote_code=True,
        torch_dtype=torch.float16 
    )
    model.eval()

    test_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    model, test_loader = accelerator.prepare(model, test_loader)

    modes = ["0-shot", "1-shot", "3-shot"]
    results = {}

    for mode in modes:
        accelerator.print(f"\n========== Running {mode.upper()} Evaluation ==========")
        all_preds = []
        all_labels = []
        
        total_batches = len(test_loader)
        
        for step, batch in enumerate(test_loader, 1):
            prompts = [build_prompt(p, h, mode) for p, h in zip(batch["premise"], batch["hypothesis"])]
            
            inputs = tokenizer(prompts, padding=True, return_tensors="pt").to(accelerator.device)
            prompt_length = inputs["input_ids"].shape[1]

            with torch.no_grad():
                # --- THE FIX: Unwrap the model to access the .generate() method ---
                unwrapped_model = accelerator.unwrap_model(model)
                outputs = unwrapped_model.generate(
                    **inputs,
                    max_new_tokens=4,
                    pad_token_id=tokenizer.eos_token_id,
                    do_sample=False, 
                    temperature=None,
                    top_p=None
                )
            
            generated_tokens = outputs[:, prompt_length:]
            decoded_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

            batch_preds = []
            for text in decoded_texts:
                text = text.lower().strip()
                if "entailment" in text: batch_preds.append(0)
                elif "neutral" in text: batch_preds.append(1)
                elif "contradiction" in text: batch_preds.append(2)
                else: batch_preds.append(1) 

            preds_tensor = torch.tensor(batch_preds, device=accelerator.device)
            labels_tensor = batch["label"].clone().detach().to(accelerator.device)
            
            preds_gathered, labels_gathered = accelerator.gather_for_metrics((preds_tensor, labels_tensor))
            
            all_preds.extend(preds_gathered.cpu().numpy().tolist())
            all_labels.extend(labels_gathered.cpu().numpy().tolist())
            
            if step % 20 == 0 and accelerator.is_main_process:
                 accelerator.print(f"[{mode.upper()}] Processed {step}/{total_batches} batches...")

        if accelerator.is_main_process:
            acc = accuracy_score(all_labels, all_preds)
            results[mode] = acc
            accelerator.print(f"--> {mode} Final Accuracy: {acc:.4f}\n")
            
            accelerator.log({
                f"baseline_accuracy/{mode}": acc
            })
            
            if mode == "3-shot":
                accelerator.print("=== 3-SHOT DETAILED REPORT ===")
                accelerator.print(classification_report(all_labels, all_preds, target_names=["entailment", "neutral", "contradiction"]))

    if accelerator.is_main_process:
        accelerator.print("\n=== FINAL BASELINE SUMMARY ===")
        for m, a in results.items():
            accelerator.print(f"{m}: {a:.4f}")
        accelerator.print(f"Fine-Tuned Classifier: 0.8620 (For Comparison)")

    accelerator.end_training()

if __name__ == "__main__":
    main()

Writing eval_nshot_baseline.py


In [7]:
!accelerate launch --multi_gpu --num_processes=2 eval_nshot_baseline.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
GPUs active: 2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kapilsandhu2022 (kapilsandhu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run mct91g63 (0.2s)
wandb: ⣾ setting up run mct91g63 (0.2s)
wandb: ⣷ setting up run mct91g63 (0.2s)
wandb: ⣯ setting up run mct91g63 (0.2s)
wandb: ⣟ setting up run mct91g63 (0.2s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260526_204304-mct91g63
wandb: R